# Intrinsic Benchmarks - Reliability

This notebook computes and visualizes the reliability intrinsic benchmarking metrics for the connectivity matrices of each of the PySPI and skarf methods. This includes batch ICC, gradient canonical correlation, subject identifiability index, & discriminability.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from joblib import Parallel, delayed

PROJECT_ROOT = Path("/home/jpillai/projects/skarf-experiments")
# sys.path.insert(0, str(PROJECT_ROOT / "src"))

from arfcexp.matrices import (
    apply_sparsity,
    load_avg_mats_and_impose_sparsity,
    EfficientMatrixReader,
)

from arfcexp.reliability import (
    build_icc_input,
    compute_gradient_reliability,
    compute_icc,
    compute_identifiability,
    compute_discriminability,
)

# paths
PARQUET_PATH = Path("/srv/projects/skarf/data_aggregation/hcp_1200_rfmri_schaefer.parquet")
SUBJECT_LIST = PROJECT_ROOT / "resources/subject_lists/hcp_complete_data_867_subject_list.txt"
OUT_DIR = PROJECT_ROOT / "results/intrinsic_benchmarks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# run params
GRADIENT_KWARGS = dict(affinity_threshold=0.8, n_components=2)

# subjects
with open(SUBJECT_LIST) as f:
    SUB_LIST = [line.strip() for line in f if line.strip()]
SUB_SET = set(SUB_LIST)

READER = EfficientMatrixReader(PARQUET_PATH)

print(f"Subjects : {len(SUB_LIST)}")
print(f"Output : {OUT_DIR}")

# plotting colors
PYSPI_COLOR = "#1f77b4"
SKARF_LAG0 = "#ff7f0e"
SKARF_LAG1 = "#ffbb78"

FLIER = dict(marker="o", markerfacecolor="none", markeredgecolor="black",
             markersize=2, alpha=0.4, linestyle="none")
MEDIAN = dict(color="black", linewidth=1.2)
BOX  = dict(linewidth=0.8)
WHISKER = dict(linewidth=0.8)
CAP = dict(linewidth=0.8)

Subjects : 867
Output   : /home/jpillai/projects/skarf-experiments/results/intrinsic_benchmarks


In [3]:
schema = pl.scan_parquet(PARQUET_PATH).schema
has_lag = "lag" in schema

select_cols = ["method", "func"] + (["lag"] if has_lag else [])
combos = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("success"))
    .select(select_cols)
    .unique()
    .collect()
    .to_pandas()
)
if not has_lag:
    combos["lag"] = None

combos = combos.reset_index(drop=True)
print(f"{len(combos)} (method, func, lag) combinations found")
combos

173 (method, func, lag) combinations found


/tmp/ipykernel_1636655/2597384383.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = pl.scan_parquet(PARQUET_PATH).schema


,method,func,lag
0,pyspi,je_kernel_W-0.5,NaN
1,pyspi,coint_aeg_tstat_trend-c_autolag-aic_maxlag-10,NaN
2,skarf,linear_lasso-pos,1.0
3,pyspi,ppc_multitaper_max_fs-1_fmin-0_fmax-0-25,NaN
4,pyspi,cov_GraphicalLassoCV,NaN
...,...,...,...
168,pyspi,pli_multitaper_max_fs-1_fmin-0_fmax-0-25,NaN
169,skarf,prec_empirical,0.0
170,pyspi,wpli_multitaper_mean_fs-1_fmin-0_fmax-0-25,NaN
171,pyspi,kendalltau,NaN


In [ ]:
def _load_run_df(method, func, lag):
    """Load raw per-run rows for one (method, func, lag) combo."""
    lag_val = None if pd.isna(lag) else int(lag)
    filters = {"method": method, "func": func, "success": True}
    if lag_val is not None:
        filters["lag"] = lag_val
    df_pl = READER.query(columns=["sub", "ses", "run", "mat"], **filters)
    return df_pl.to_pandas()

### ICC

In [ ]:
def process_icc_combo(combo_idx, combos, sub_set):
    row = combos.iloc[combo_idx]
    method = row["method"]
    func = row["func"]
    lag = row["lag"]

    run_df = _load_run_df(method, func, lag)
    run_df = run_df[run_df["sub"].astype(str).isin(sub_set)]

    if run_df.empty:
        return {"method": method, "func": func, "lag": lag,
                "mean_icc2": np.nan, "mean_icc3": np.nan, "icc2": None}

    result = compute_icc(run_df)

    return {
        "method": method,
        "func": func,
        "lag": lag,
        "mean_icc2": float(np.nanmean(result.icc2)),
        "mean_icc3": float(np.nanmean(result.icc3)),
        "icc2": result.icc2,
    }

In [ ]:
# print(f"Computing ICC for {len(combos)} combos...")

# icc_results = [
#     process_icc_combo(i, combos, PARQUET_PATH, SUB_SET)
#     for i in range(len(combos))
# ]

# icc_summary_df  = pd.DataFrame([{k: v for k, v in r.items() if k != "icc2"} for r in icc_results])
# icc_edgewise_df = pd.DataFrame(
#     [{"method": r["method"], "func": r["func"], "lag": r["lag"], "icc2": r["icc2"]}
#      for r in icc_results if r["icc2"] is not None]
# )

# icc_summary_df.to_parquet(OUT_DIR / "icc_summary.parquet", index=False)
# icc_edgewise_df.to_parquet(OUT_DIR / "icc_edgewise.parquet", index=False)

# print(f"\icc_summary.parquet  — {len(icc_summary_df)} rows")
# print(f"✓ icc_edgewise.parquet — {len(icc_edgewise_df)} rows")
# icc_summary_df

In [ ]:
#FIX PLOTS & INPUTS: use Alp's structure
icc_edgewise_df = pd.read_parquet(OUT_DIR / "icc_edgewise.parquet")

# icc2 arrays into one row per edge
records = []
for _, row in icc_edgewise_df.iterrows():
    if row["icc2"] is None:
        continue
    for val in row["icc2"]:
        records.append({
            "func": row["func"],
            "method": row["method"],
            "lag": row["lag"],
            "icc2": val,
        })
edges_df = pd.DataFrame(records)
edges_df = edges_df[np.isfinite(edges_df["icc2"])]

pyspi_df = edges_df[edges_df["method"] == "pyspi"].copy()
skarf_df = edges_df[edges_df["method"] == "skarf"].copy()

pyspi_order = (
    pyspi_df.groupby("func")["icc2"].median()
    .sort_values(ascending=False).index.tolist()
)
skarf_order = (
    skarf_df.groupby(["func", "lag"])
    .apply(lambda g: g["icc2"].median())
    .reset_index()
    .groupby("func")[0].mean()
    .sort_values(ascending=False).index.tolist()
)


fig, (ax_p, ax_s) = plt.subplots(
    1, 2,
    figsize=(max(len(pyspi_order) * 0.35 + 2, 10),
             max(len(skarf_order) * 0.5 + 2, 6)),
    sharey=True,
    gridspec_kw={"width_ratios": [len(pyspi_order), len(skarf_order)]}
)

# pyspi panel
pyspi_data = [pyspi_df[pyspi_df["func"] == f]["icc2"].dropna().values
              for f in pyspi_order]

bp = ax_p.boxplot(
    pyspi_data,
    positions=range(len(pyspi_order)),
    widths=0.5,
    patch_artist=True,
    flierprops=FLIER,
    medianprops=MEDIAN,
    boxprops=BOX,
    whiskerprops=WHISKER,
    capprops=CAP,
    showfliers=True,
)
for patch in bp["boxes"]:
    patch.set_facecolor(PYSPI_COLOR)
    patch.set_alpha(0.7)

ax_p.set_xticks(range(len(pyspi_order)))
ax_p.set_xticklabels(pyspi_order, rotation=90, ha="right", fontsize=6)
ax_p.set_ylabel("ICC2")
ax_p.set_title("(a) PySPI metrics (0.8 sparsity)")
ax_p.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.4)
ax_p.set_ylim(-0.2, 1.0)

# skarf panel
lags = sorted(skarf_df["lag"].dropna().unique())
n_lags = len(lags)
width = 0.35
lag_colors = {lags[0]: SKARF_LAG0, lags[1]: SKARF_LAG1} if n_lags > 1 else {lags[0]: SKARF_LAG0}
lag_hatch  = {lags[0]: "",          lags[1]: "///"}      if n_lags > 1 else {lags[0]: ""}

for lag_idx, lag in enumerate(lags):
    lag_sub = skarf_df[skarf_df["lag"] == lag]
    offsets = [i + (lag_idx - (n_lags - 1) / 2) * width for i in range(len(skarf_order))]
    data = [lag_sub[lag_sub["func"] == f]["icc2"].dropna().values for f in skarf_order]

    bp = ax_s.boxplot(
        data,
        positions=offsets,
        widths=width * 0.85,
        patch_artist=True,
        flierprops=FLIER,
        medianprops=MEDIAN,
        boxprops=BOX,
        whiskerprops=WHISKER,
        capprops=CAP,
        showfliers=True,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(lag_colors[lag])
        patch.set_hatch(lag_hatch[lag])
        patch.set_alpha(0.7)

ax_s.set_xticks(range(len(skarf_order)))
ax_s.set_xticklabels(skarf_order, rotation=90, ha="right", fontsize=7)
ax_s.set_title("(b) skarf metrics (0.8 sparsity)")
ax_s.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.4)

legend_elements = [
    mpatches.Patch(facecolor=PYSPI_COLOR, alpha=0.7, label="pyspi"),
    mpatches.Patch(facecolor=SKARF_LAG0,  alpha=0.7, label="skarf lag 0"),
    mpatches.Patch(facecolor=SKARF_LAG1,  alpha=0.7, hatch="///", label="skarf lag 1"),
]
fig.legend(handles=legend_elements, loc="upper center", ncol=3,
           frameon=True, fontsize=8, bbox_to_anchor=(0.5, 1.02))

plt.suptitle("Edge-wise ICC2", y=1.04, fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / "icc2_edgewise_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

### Gradient Canonical Correlation

Mean pairwise canonical correlation between principal gradients (Laplacian Eigenmaps, 
k=2 components) computed across all run pairs per subject. For each subject and method, 
gradients are estimated from a sparse affinity matrix (quantile threshold=0.8) for every 
run, then all pairs of runs are compared using subspace canonical correlation - defined 
as the mean squared cosine of principal angles. The per-subject similarity scores are 
then aggregated across subjects. Higher values indicate that the low-dimensional 
organisation of the connectivity matrix is reproducible across scanning sessions, 
even when individual edge weights are not.

In [ ]:
def process_gradient_combo(combo_idx, combos, sub_set, gradient_kwargs):
    row = combos.iloc[combo_idx]
    method = row["method"]
    func = row["func"]
    lag = row["lag"]

    run_df = _load_run_df(method, func, lag)
    run_df = run_df[run_df["sub"].astype(str).isin(sub_set)]

    if run_df.empty:
        return pd.DataFrame()

    df = compute_gradient_reliability(run_df, **gradient_kwargs)
    df["method"] = method
    df["func"] = func
    df["lag"] = lag
    return df

In [ ]:
# print(f"Computing gradient reliability for {len(combos)} combos...")

# grad_results = [
#     process_gradient_combo(i, combos, PARQUET_PATH, SUB_SET, GRADIENT_KWARGS)
#     for i in range(len(combos))
# ]

# gradient_df = pd.concat([df for df in grad_results if not df.empty], ignore_index=True)
# gradient_df.to_parquet(OUT_DIR / "gradient_reliability.parquet", index=False)

# print(f"\gradient_reliability.parquet — {len(gradient_df)} rows")
# gradient_df.groupby(["method", "func", "lag"])["gradient_similarity"].mean().reset_index()

Computing gradient reliability for 173 combos using 1 jobs...


/home/jpillai/projects/skarf-experiments/.venv/lib/python3.11/site-packages/brainspace/gradient/embedding.py:211: UserWarning: Graph is not fully connected.
  warnings.warn('Graph is not fully connected.')
/home/jpillai/projects/skarf-experiments/.venv/lib/python3.11/site-packages/brainspace/gradient/embedding.py:211: UserWarning: Graph is not fully connected.
  warnings.warn('Graph is not fully connected.')
/home/jpillai/projects/skarf-experiments/.venv/lib/python3.11/site-packages/brainspace/gradient/embedding.py:211: UserWarning: Graph is not fully connected.
  warnings.warn('Graph is not fully connected.')
/home/jpillai/projects/skarf-experiments/.venv/lib/python3.11/site-packages/brainspace/gradient/embedding.py:211: UserWarning: Graph is not fully connected.
  warnings.warn('Graph is not fully connected.')
/home/jpillai/projects/skarf-experiments/.venv/lib/python3.11/site-packages/brainspace/gradient/embedding.py:211: UserWarning: Graph is not fully connected.
  warnings.warn('Gr


✓ gradient_reliability.parquet — 149133 rows


,method,func,lag,gradient_similarity
0,skarf,cov_empirical,0.0,0.660142
1,skarf,cov_empirical,1.0,0.567295
2,skarf,cov_graphicallasso,0.0,0.407108
3,skarf,cov_graphicallasso,1.0,0.347990
4,skarf,linear_enet,0.0,0.713501
5,skarf,linear_enet,1.0,0.617015
6,skarf,linear_enet-pos,0.0,0.507009
7,skarf,linear_enet-pos,1.0,0.447825
8,skarf,linear_lasso,0.0,0.693283
9,skarf,linear_lasso,1.0,0.597967


In [ ]:
#FIX PLOTS & INPUTS: use Alp's structure
gradient_df = pd.read_parquet(OUT_DIR / "gradient_reliability.parquet")
gradient_df = gradient_df[np.isfinite(gradient_df["gradient_similarity"])]

pyspi_g = gradient_df[gradient_df["method"] == "pyspi"].copy()
skarf_g = gradient_df[gradient_df["method"] == "skarf"].copy()

pyspi_order = (
    pyspi_g.groupby("func")["gradient_similarity"].median()
    .sort_values(ascending=False).index.tolist()
)
skarf_order = (
    skarf_g.groupby("func")["gradient_similarity"].median()
    .sort_values(ascending=False).index.tolist()
)

fig, (ax_p, ax_s) = plt.subplots(
    1, 2,
    figsize=(max(len(pyspi_order) * 0.35 + 2, 10),
             max(len(skarf_order) * 0.5 + 2, 6)),
    sharey=True,
    gridspec_kw={"width_ratios": [len(pyspi_order), len(skarf_order)]}
)

FLIER = dict(marker="o", markerfacecolor="none", markeredgecolor="black",
               markersize=2, alpha=0.4, linestyle="none")
MEDIAN = dict(color="black", linewidth=1.2)
BOX = dict(linewidth=0.8)
WHISKER = dict(linewidth=0.8)
CAP = dict(linewidth=0.8)

# pyspi panel
pyspi_data = [pyspi_g[pyspi_g["func"] == f]["gradient_similarity"].dropna().values
              for f in pyspi_order]

bp = ax_p.boxplot(
    pyspi_data,
    positions=range(len(pyspi_order)),
    widths=0.5,
    patch_artist=True,
    flierprops=FLIER, medianprops=MEDIAN,
    boxprops=BOX, whiskerprops=WHISKER, capprops=CAP,
)
for patch in bp["boxes"]:
    patch.set_facecolor(PYSPI_COLOR)
    patch.set_alpha(0.7)

ax_p.set_xticks(range(len(pyspi_order)))
ax_p.set_xticklabels(pyspi_order, rotation=90, ha="right", fontsize=6)
ax_p.set_ylabel("Gradient similarity")
ax_p.set_title("(a) PySPI metrics (0.8 sparsity)")
ax_p.set_ylim(0, 1.05)
ax_p.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.4)

# skarf panel
lags = sorted(skarf_g["lag"].dropna().unique())
n_lags = len(lags)
width = 0.35
lag_colors = {lags[0]: SKARF_LAG0, lags[1]: SKARF_LAG1} if n_lags > 1 else {lags[0]: SKARF_LAG0}
lag_hatch  = {lags[0]: "",          lags[1]: "///"}      if n_lags > 1 else {lags[0]: ""}

for lag_idx, lag in enumerate(lags):
    lag_sub = skarf_g[skarf_g["lag"] == lag]
    offsets = [i + (lag_idx - (n_lags - 1) / 2) * width for i in range(len(skarf_order))]
    data = [lag_sub[lag_sub["func"] == f]["gradient_similarity"].dropna().values
            for f in skarf_order]

    bp = ax_s.boxplot(
        data,
        positions=offsets,
        widths=width * 0.85,
        patch_artist=True,
        flierprops=FLIER, medianprops=MEDIAN,
        boxprops=BOX, whiskerprops=WHISKER, capprops=CAP,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(lag_colors[lag])
        patch.set_hatch(lag_hatch[lag])
        patch.set_alpha(0.7)

ax_s.set_xticks(range(len(skarf_order)))
ax_s.set_xticklabels(skarf_order, rotation=90, ha="right", fontsize=7)
ax_s.set_title("(b) skarf metrics (0.8 sparsity)")

legend_elements = [
    mpatches.Patch(facecolor=PYSPI_COLOR, alpha=0.7, label="pyspi"),
    mpatches.Patch(facecolor=SKARF_LAG0,  alpha=0.7, label="skarf lag 0"),
    mpatches.Patch(facecolor=SKARF_LAG1,  alpha=0.7, hatch="///", label="skarf lag 1"),
]
fig.legend(handles=legend_elements, loc="upper center", ncol=3,
           frameon=True, fontsize=8, bbox_to_anchor=(0.5, 1.02))

plt.suptitle("Gradient canonical correlation", y=1.04, fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / "gradient_reliability_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

### Subject Identifiability Index

Reference: https://www.nature.com/articles/s41598-018-25089-1

In [ ]:
def process_sii_combo(combo_idx, combos, sub_set):
    row = combos.iloc[combo_idx]
    method = row["method"]
    func = row["func"]
    lag = row["lag"]

    run_df = _load_run_df(method, func, lag)
    run_df = run_df[run_df["sub"].astype(str).isin(sub_set)]

    if run_df.empty:
        return {"method": method, "func": func, "lag": lag,
                "I_diff": np.nan, "success_rate": np.nan}

    result = compute_identifiability(run_df)

    return {
        "method": method,
        "func": func,
        "lag": lag,
        "I_diff": result["I_diff"],
        "success_rate": result["success_rate"],
    }

In [ ]:
# print(f"Computing SII for {len(combos)} combos...")

# sii_results = [
#     process_sii_combo(i, combos, PARQUET_PATH, SUB_SET)
#     for i in range(len(combos))
# ]

# sii_df = pd.DataFrame(sii_results)
# sii_df.to_parquet(OUT_DIR / "sii.parquet", index=False)
# print(f"\nsii.parquet — {len(sii_df)} rows")
# sii_df

Computing SII for 173 combos...

sii.parquet — 173 rows


,method,func,lag,I_diff,success_rate
0,pyspi,je_kernel_W-0.5,NaN,0.000000,0.001153
1,pyspi,coint_aeg_tstat_trend-c_autolag-aic_maxlag-10,NaN,0.000000,0.001153
2,skarf,linear_lasso-pos,1.0,13.464409,0.990773
3,pyspi,ppc_multitaper_max_fs-1_fmin-0_fmax-0-25,NaN,0.000000,0.001153
4,pyspi,cov_GraphicalLassoCV,NaN,0.000000,0.001164
...,...,...,...,...,...
168,pyspi,pli_multitaper_max_fs-1_fmin-0_fmax-0-25,NaN,1.406896,0.088812
169,skarf,prec_empirical,0.0,0.619631,0.005767
170,pyspi,wpli_multitaper_mean_fs-1_fmin-0_fmax-0-25,NaN,13.278643,0.420992
171,pyspi,kendalltau,NaN,0.000000,0.001153


In [ ]:
#FIX PLOTS & INPUTS: use Alp's structure
sii_df = pd.read_parquet(OUT_DIR / "sii.parquet")

sii_df["lag_str"] = sii_df["lag"].apply(lambda x: "NA" if pd.isna(x) else str(int(float(x))))
sii_df["label"] = sii_df.apply(
    lambda r: r["func"] if r["lag_str"] == "NA" else f"{r['func']} (lag {r['lag_str']})",
    axis=1
)
sii_df = sii_df.sort_values("I_diff", ascending=True).reset_index(drop=True)

n = len(sii_df)
fig, ax = plt.subplots(figsize=(7, n * 0.22 + 1))

for i, row in sii_df.iterrows():
    color = (PYSPI_COLOR if row["lag_str"] == "NA"
             else SKARF_LAG0 if row["lag_str"] == "0"
             else SKARF_LAG1)
    hatch = "///" if row["lag_str"] == "1" else ""
    ax.barh(i, row["I_diff"], color=color, hatch=hatch,
            alpha=0.7, edgecolor="black" if hatch else "none", linewidth=0.5)

ax.set_yticks(range(n))
ax.set_yticklabels(sii_df["label"], fontsize=6)
ax.set_xlabel("I_diff")
ax.set_title("Subject Identifiability Index (I_diff)")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

legend_elements = [
    mpatches.Patch(facecolor=PYSPI_COLOR, alpha=0.7, label="pyspi"),
    mpatches.Patch(facecolor=SKARF_LAG0,  alpha=0.7, label="skarf lag 0"),
    mpatches.Patch(facecolor=SKARF_LAG1,  alpha=0.7, hatch="///",
                   edgecolor="black", label="skarf lag 1"),
]
ax.legend(handles=legend_elements, loc="lower right", frameon=True, fontsize=8)

sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / "sii_idiff.png", dpi=300, bbox_inches="tight")
plt.show()

### Discriminability

In [ ]:
def process_discriminability_combo(combo_idx, combos, sub_set):
    row = combos.iloc[combo_idx]
    method = row["method"]
    func = row["func"]
    lag = row["lag"]

    run_df = _load_run_df(method, func, lag)
    run_df = run_df[run_df["sub"].astype(str).isin(sub_set)]

    if run_df.empty:
        return {"method": method, "func": func, "lag": lag, "discriminability": np.nan}

    d = compute_discriminability(run_df)

    print(f"{method}/{func}/lag={lag} — discriminability={d:.3f}")
    return {"method": method, "func": func, "lag": lag, "discriminability": d}

In [ ]:
# discr_results = [
#     process_discriminability_combo(i, combos, PARQUET_PATH, SUB_SET)
#     for i in range(len(combos))
# ]

# discr_df = pd.DataFrame(discr_results)
# discr_df.to_parquet(OUT_DIR / "discriminability.parquet", index=False)
# print(f"\ndiscriminability.parquet — {len(discr_df)} rows")
# discr_df